# Extracción de Sumillas de Cursos (PDFs)

Extracción sencilla de **carrera**, **curso** y **descripción** desde los 4 PDFs de sumillas usando `pdftotext` y `pandas`.

In [ ]:
import glob
import os
import subprocess
import pandas as pd

In [ ]:
CARRERAS = {
    "ingenieria-electronica-sumillas.pdf": "Ingeniería Electrónica",
    "ingenieria-industrial-sumillas.pdf": "Ingeniería Industrial",
    "ingenieria-informatica-sumillas.pdf": "Ingeniería Informática",
    "ingenieria-mecatronica-sumillas.pdf": "Ingeniería Mecatrónica",
}

IGNORAR = ("FACULTAD", "ESCUELA", "SUMILLA", "PLAN DE", "SEMESTRE", "ELECTIVOS")


def extraer_cursos(ruta_pdf):
    nombre_archivo = os.path.basename(ruta_pdf)
    carrera = CARRERAS.get(nombre_archivo, nombre_archivo)
    texto = subprocess.check_output(["pdftotext", ruta_pdf, "-"], text=True)

    filas = []
    curso_actual = None
    desc_actual = []

    for linea in texto.splitlines():
        l = linea.replace("\x0c", "").strip()
        if not l or l.isdigit():
            continue
        if any(p in l.upper() for p in IGNORAR) and len(l) < 60:
            continue

        # Los títulos de curso están en MAYÚSCULAS y no terminan en punto
        if l == l.upper() and sum(c.isalpha() for c in l) >= 4 and not l.endswith("."):
            if curso_actual:
                filas.append({
                    "carrera": carrera,
                    "curso": curso_actual,
                    "descripcion": " ".join(desc_actual),
                })
            curso_actual = l.rstrip(":")
            desc_actual = []
        elif curso_actual:
            desc_actual.append(l)

    if curso_actual:
        filas.append({
            "carrera": carrera,
            "curso": curso_actual,
            "descripcion": " ".join(desc_actual),
        })

    return filas

In [ ]:
todos_los_cursos = []
for pdf in sorted(glob.glob("*sumillas.pdf")):
    todos_los_cursos.extend(extraer_cursos(pdf))

df = pd.DataFrame(todos_los_cursos)
print(df["carrera"].value_counts())
df.head()

In [ ]:
# Guardar resultado para el análisis de habilidades
df.to_csv("sumillas_cursos.csv", index=False)
print(f"Guardados {len(df)} cursos en sumillas_cursos.csv")

## Emparejamiento de habilidades (fuerza bruta)

In [ ]:
import re

# Lista predefinida: habilidad -> variantes a buscar en la descripción
HABILIDADES = {
    # Programación y Datos
    "POO": ["poo", "programación orientada a objetos", "orientada a objetos"],
    "SQL / Bases de Datos": ["sql", "base de datos", "bases de datos", "mysql", "postgresql", "oracle", "modelamiento de datos"],
    "NoSQL": ["nosql", "mongodb"],
    "Python": ["python"],
    "Java": ["java"],
    "C/C++": ["c++", "lenguaje c"],
    "JavaScript / Web": ["javascript", "html", "css", "desarrollo web", "aplicaciones web"],
    "Desarrollo Móvil": ["móvil", "móviles", "android"],
    "Algoritmos y Estructuras": ["algoritmos", "estructuras de datos"],
    "Git": ["git", "control de versiones"],
    "Linux": ["linux", "unix"],
    "Sistemas Operativos": ["sistemas operativos", "sistema operativo"],
    "Cloud Computing": ["cloud", "nube", "aws", "azure"],
    "Big Data / Minería de Datos": ["big data", "minería de datos", "data mining", "ciencia de datos"],
    "Machine Learning / IA": ["machine learning", "aprendizaje automático", "inteligencia artificial", "redes neuronales"],
    "Ciberseguridad": ["ciberseguridad", "seguridad informática", "seguridad de la información", "criptografía"],
    "Redes": ["redes", "tcp/ip", "cisco", "enrutamiento", "conmutación"],
    # Electrónica, Control y Mecatrónica
    "MATLAB / Simulink": ["matlab", "simulink"],
    "AutoCAD / CAD": ["autocad", "cad", "solidworks", "diseño asistido"],
    "PLC / Automatización": ["plc", "autómatas programables", "automatización", "scada"],
    "Arduino": ["arduino"],
    "Sistemas Embebidos": ["embebidos", "microcontroladores", "microprocesadores"],
    "FPGA / VHDL": ["fpga", "vhdl"],
    "Control Automático": ["control automático", "sistemas de control", "pid"],
    "Sensores e Instrumentación": ["sensores", "actuadores", "instrumentación"],
    "Neumática e Hidráulica": ["neumática", "hidráulica", "electroneumática"],
    "Electrónica": ["electrónica", "circuitos"],
    "Telecomunicaciones": ["telecomunicaciones", "señales", "antenas", "fibra óptica"],
    "IoT": ["iot", "internet de las cosas"],
    "Robótica": ["robótica", "robots"],
    # Industrial, Gestión y Analítica
    "Excel": ["excel"],
    "Power BI": ["power bi", "inteligencia de negocios", "business intelligence"],
    "ERP / SAP": ["erp", "sap"],
    "Estadística": ["estadística", "probabilidades"],
    "Investigación de Operaciones": ["investigación de operaciones", "programación lineal", "optimización"],
    "Lean / Six Sigma": ["lean", "six sigma", "mejora continua"],
    "Gestión de Calidad": ["gestión de calidad", "calidad total", "iso 9001"],
    "Logística / Supply Chain": ["logística", "cadena de suministro", "supply chain", "inventarios"],
    "Gestión de Proyectos": ["gestión de proyectos", "pmbok", "scrum", "ágil", "ágiles"],
    "Seguridad Industrial": ["seguridad industrial", "salud ocupacional", "sst"],
    "Costos y Finanzas": ["costos", "presupuestos", "evaluación de proyectos", "finanzas"],
    # Transversales e Idiomas
    "Inglés": ["inglés"],
    "Trabajo en Equipo": ["trabajo en equipo"],
    "Liderazgo": ["liderazgo"],
    "Comunicación Efectiva": ["comunicación efectiva", "comunicación oral y escrita"],
}


def buscar_habilidades(texto):
    texto = str(texto).lower()
    encontradas = []
    for hab, variantes in HABILIDADES.items():
        for v in variantes:
            if re.search(rf"\b{re.escape(v)}\b", texto):
                encontradas.append(hab)
                break
    return encontradas


df["habilidades"] = df["descripcion"].apply(buscar_habilidades)
df[df["habilidades"].str.len() > 0][["carrera", "curso", "habilidades"]].head(10)

In [ ]:
# Ver frecuencia de habilidades por carrera
df_hab = df.explode("habilidades").dropna(subset=["habilidades"])
print(df_hab["habilidades"].value_counts())

df.to_csv("sumillas_habilidades.csv", index=False)